In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict


In [25]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
}

DATASETS_MT = [
    'wmt14_deen',
    'wmt14_fren',
    'wmt14_csen',
    'wmt14_ruen',
    'wmt19_ruen',
    'wmt19_fien',
    'wmt19_deen',
    'wmt19_lten'
]

all_metrics_mt = ['Comet-wmt22-comet-da', 'XComet-XCOMET-XXL', 'metricx-metricx-24-hybrid-large-v2p6']
all_methods =['MSP', 'PPL', 'MTE', 'MCSE', 'MCNSE']


metrics_dict ={
  'Comet':'Comet', 'XComet-XCOMET-XXL' :'XComet-XXL', 'metricx-metricx-24-hybrid-large-v2p6' :'MetricX-Large' ,
  'AlignScoreInputOutput':'Align Score', 'Accuracy':'Acc', 'AlignScoreInputOutput':'Align Score','Rouge_rougeL':'Rouge L', 'Comet-wmt22-comet-da':'Comet',
    'MSP':'MSP', 'PPL' :'PPL', 'MTE' :'MTE',  'MCSE':'MCSE', 'MCNSE':'MCNSE', 'LSRL': 'LSRL'}

In [32]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import linregress
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

sns.set(style="whitegrid", font_scale=1.4, rc={"font.family": "serif"})


def format_dataset_name(raw_name):
    try:
        prefix, lang_pair = raw_name.split("_")
        prefix = prefix.upper()

        if len(lang_pair) == 4:  # e.g., fren → Fr-En
            src = lang_pair[:2].capitalize()
            tgt = lang_pair[2:].capitalize()
            lang_fmt = f"{src}-{tgt}"
        else:
            lang_fmt = lang_pair.upper()

        return f"{prefix} {lang_fmt}"
    except Exception:
        return raw_name.upper()


def plot_metric_vs_length(
    gen_lengths, metric_values,
    metric_name, dataset_name, save_path='plot.pdf', model='llama', task ='nmt'
):

    # Trim outliers
    upper_q, lower_q = np.quantile(gen_lengths, [0.95, 0.05])
    mask = (gen_lengths > lower_q) & (gen_lengths < upper_q)
    gen_lengths = gen_lengths[mask]
    metric_values = metric_values[mask]

    # Normalize
    scaler_len = MinMaxScaler()
    scaler_val = MinMaxScaler()

    norm_len = scaler_len.fit_transform(gen_lengths[:, None]).squeeze()
    norm_val = scaler_val.fit_transform(metric_values[:, None]).squeeze()

    # Bin and smooth
    df = pd.DataFrame({"length": norm_len, "metric": norm_val})
    grouped = df.groupby("length").agg(['mean', 'sem'])
    x_vals = grouped.index.values
    y_vals = grouped['metric']['mean'].values
    y_errs = grouped['metric']['sem'].values

    # Fit regression (on raw normalized data)
    linreg = LinearRegression().fit(norm_len[:, None], norm_val)
    slope = linreg.coef_[0]

    # Also compute p-value
    slope_, intercept_, r_val, p_val, std_err = linregress(norm_len, norm_val)

    x_line = np.linspace(0, 1, 100)
    y_line = linreg.predict(x_line[:, None])

    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x_vals, y_vals, label='AVG metric value', color="navy")
    ax.fill_between(x_vals, y_vals - y_errs, y_vals + y_errs, alpha=0.2, color="navy")
    ax.plot(x_line, y_line, linestyle='--', color='crimson', label='Regression Line')

    ax.text(0.05, 0.95,
            f"Slope: {slope:.2f}\n$p$-value: {p_val:.3f}",
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='gray'))

    if task=='nmt':
        pretty_dataset = format_dataset_name(dataset_name)
    else:
        pretty_dataset = dataset_name.capitalize()
    ax.set_title(f"{metrics_dict[metric_name]} vs. Length ({pretty_dataset})", fontsize=14)
    ax.set_xlabel("Generated sequence length (normalized)")
    ax.set_ylabel(f"{metrics_dict[metric_name]} (normalized)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)


# Metric trends plots (Translation)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from utils import extract_and_prepare_data
import numpy as np
import os 

models =['llama','gemma','eurollm']
for model in models:
    for dataset in DATASETS_MT:
        train_ue_values, test_ue_values, train_metric_values, test_metric_values, train_gen_lengths, gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics_mt, model=model)

        for metric in all_metrics_mt:
            os.makedirs(f'plots', exist_ok=True)

            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                save_path=f'plots/{model}_{dataset}_{metric}_train.pdf',
            )


# Metric trends plots (Summarization)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os 

DATASETS_SUM =['xsum']
all_metrics_sum = ['AlignScoreInputOutput']
models_sum =['llama','gemma']

for model in models_sum:
    for dataset in DATASETS_SUM:
        train_ue_values, test_ue_values, train_metric_values, test_metric_values, train_gen_lengths, gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics_sum, model=model)

        for metric in all_metrics_sum:
            os.makedirs(f'plots', exist_ok=True)

            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                save_path=f'plots/{dataset}_{metric}_{model}_train.pdf',
                task='sum'
            )


# Metric trends plots (Mathematical Reasoning)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os 
DATASETS_MR =['gsm8k']
all_metrics_mr=['Accuracy']
models_mr =['llama','gemma']
for model in models_mr:
    for dataset in DATASETS_MR:
        train_ue_values, test_ue_values, train_metric_values, test_metric_values, train_gen_lengths, gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics_mr, model=model)

        for metric in all_metrics_mr:
            os.makedirs(f'plots', exist_ok=True)

            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                save_path=f'plots/{dataset}_{metric}_{model}_train.pdf',
                task='mr'
            )


# UE metrics trends (Translation)

In [35]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL'
}

for model in models:
    for dataset in DATASETS_MT:
        train_ue_values, test_ue_values, train_metric_values, test_metric_values, train_gen_lengths, gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics_mt, model=model)

        for metric, metric_short  in methods_dict.items():
            os.makedirs(f'plots', exist_ok=True)
            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_ue_values[metric_short]),
                metric_name=metric_short,
                dataset_name=dataset,
                save_path=f'plots/{dataset}_{metric}_{model}_train.pdf',
            )

# UE metrics trends (Summarization)

In [ ]:

for model in models_sum:
    for dataset in DATASETS_SUM:
        train_ue_values, test_ue_values, train_metric_values, test_metric_values, train_gen_lengths, gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics_sum, model=model)

        for metric, metric_short  in methods_dict.items():
            os.makedirs(f'plots', exist_ok=True)
            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_ue_values[metric_short]),
                metric_name=metric_short,
                dataset_name=dataset,
                save_path=f'plots/{dataset}_{metric}_{model}_train.pdf',
                task='sum'
            )

# UE metrics trends (Mathematical Reasoning)

In [ ]:

for model in models_mr:
    for dataset in DATASETS_MR:
        train_ue_values, test_ue_values, train_metric_values, test_metric_values, train_gen_lengths, gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics_mr, model=model)

        for metric, metric_short  in methods_dict.items():
            os.makedirs(f'plots', exist_ok=True)
            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_ue_values[metric_short]),
                metric_name=metric_short,
                dataset_name=dataset,
                save_path=f'plots/{dataset}_{metric}_{model}_train.pdf',
                task='mr'
            )